# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

We will be using Ranking/Scoring because 

**Scoring** is the process of assigning a numerical value or probability to each data instance that indicates how likely it is to belong to a particular class, exhibit a behavior, or produce an outcome.

**Ranking** is the process of ordering data instances based on their predicted scores or relevance so that the highest-priority or most relevant items appear first.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df=pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

We use a transparent baseline: counting how many risk conditions (declining, page-one, stale) are true for each page, rather than assigning invented weights. This keeps the score fully explainable — every point traces to a specific, checkable fact about the page, not a guess. It's still a proxy (we don't have a true 'this page was refreshed and it worked' outcome), but it avoids inventing false precision.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['is_page_one'] = (df['position_tier'] == 'page_1').astype(int)
df['is_stale'] = (df['content_age_days'] >= 180).astype(int)
df['opportunity_flags'] = df['is_declining'] + df['is_page_one'] + df['is_stale']
def reason_codes(row):
    reasons = []
    if row['is_declining']: reasons.append('declining traffic')
    if row['is_page_one']: reasons.append('page-one decay risk')
    if row['is_stale']: reasons.append('stale content')
    return ', '.join(reasons) if reasons else 'no major flags'
df['reason_code'] = df.apply(reason_codes, axis=1)
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining,is_page_one,is_stale,opportunity_flags,reason_code
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.0,good,striking,down,-41.4,1,0,1,2,"declining traffic, stale content"
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.0,good,page_3_5,down,-57.7,1,0,1,2,"declining traffic, stale content"
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.0,good,page_3_5,down,-60.9,1,0,0,1,declining traffic
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.0,good,page_1,stable,-13.8,0,1,1,2,"page-one decay risk, stale content"
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.0,good,page_3_5,down,-34.7,1,0,1,2,"declining traffic, stale content"


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric: Precision@K (Precision@15)**

Why this metric?
Standard classification accuracy is not useful here because an SEO editor cannot review thousands of pages. If an editor only has bandwidth to review 15 pages a week, Precision@15 measures what percentage of the top 15 recommended pages were actually dying, high-value pages worth their time.

What number means 'good'?
A Precision@15 score above 0.80 (80%) means at least 12 out of the top 15 pages handed to the team are urgent, high-impact refresh candidates.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_15_queue = df.sort_values(by='opportunity_score', ascending=False).head(15)
print("Top 15 Pages in the Refresh Queue:")
print(top_15_queue[['content_id', 'cpc', 'avg_position', 'trend_direction', 'opportunity_score']])


KeyError: 'opportunity_score'

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: 1 row = 1 unique webpage (identified by content_id).

The dataframe below demonstrates that each row captures the metrics, engagement signals, search performance, and constructed opportunity score for a single content piece.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show unit of analysis as a clean slice
cols_to_show = ['content_id', 'main_intent','cpc','avg_position','trend_direction','opportunity_score']
print(f"Unit of Analysis DataFrame Shape: {df[cols_to_show].head().shape}")
df[cols_to_show].head()

Unit of Analysis DataFrame Shape: (5, 6)


,content_id,main_intent,cpc,avg_position,trend_direction,opportunity_score
0,content_304f48230142,transactional,2.05,10.6,down,59.8
1,content_a1fb4e703a9e,informational,0.05,20.3,down,74.0
2,content_9aa793d4d895,informational,0.00,36.5,down,53.2
3,content_331d6c4de07b,commercial,0.00,6.2,stable,35.2
4,content_d99b7a2d90ca,informational,0.00,44.0,down,52.4


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-else rule (e.g., if age > 180 and trend == 'down') fails in practice because:

It lacks nuance: A 170-day-old page losing 50,000 visits on a high-CPC keyword might be ignored by a strict 180-day rule, while a 181-day-old page losing only 2 visits gets flagged.

Multi-variable trade-offs: Ranking position, traffic decline, engagement rates, and search value interact in complex, non-linear ways. Hardcoded if-else logic quickly becomes unmaintainable, whereas ML dynamically weights competing signals simultaneously to produce an optimal review queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule_based = df[(df['trend_direction'] == 'down') & (df['position_tier'] == 'page_1')]
print(f"Total pages flagged by basic rule: {len(rule_based)}")
print(f"Total pages in dataset: {len(df)}")
print("ML allows us to continuously score and rank all pages instead of a binary cut-off.")

Total pages flagged by basic rule: 6730
Total pages in dataset: 30000
ML allows us to continuously score and rank all pages instead of a binary cut-off.


## Self-check

Before you submit, confirm each line honestly:

- [ yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes ] No client names, URLs, or private queries anywhere
- [ yes ] My claims use careful words: observed, measured, directional, decision-support
- [ yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.